## Data Cleaning & Preprocessing 
Fixes encoding artifacts, parses ratings/time fields, removes duplicates and unusable rows, flags Google's auto-generated rating tags (no free text), and derives a sentiment label from the star rating.

In [1]:
import re
import pandas as pd
import numpy as np

RAW_PATH = "DATASET.csv"
OUT_PATH = "cleaned_reviews.csv"

df = pd.read_csv(RAW_PATH, encoding="cp1252")
print("Raw shape:", df.shape)

# ---------------------------------------------------------------
# 1. Fix corrupted / mis-encoded store name (e.g. "ýýýMcDonald's")
# ---------------------------------------------------------------
df["store_name"] = df["store_name"].astype(str).str.replace(
    r"^[^A-Za-z0-9]+", "", regex=True
).str.strip()

# ---------------------------------------------------------------
# 2. Convert star-rating text ("1 star" / "5 stars") -> integer 1-5
# ---------------------------------------------------------------
df["rating_num"] = df["rating"].astype(str).str.extract(r"(\d)").astype(int)

# ---------------------------------------------------------------
# 3. Convert rating_count "1,240" -> integer
# ---------------------------------------------------------------
df["rating_count_num"] = (
    df["rating_count"].astype(str).str.replace(",", "", regex=False).astype(int)
)

# ---------------------------------------------------------------
# 4. Convert relative review_time ("3 months ago") -> approx days ago
# ---------------------------------------------------------------
def relative_to_days(text):
    text = str(text).lower().strip()
    m = re.match(r"(a|an|\d+)\s+(day|week|month|year|hour|minute)s?\s+ago", text)
    if not m:
        return np.nan
    qty, unit = m.groups()
    qty = 1 if qty in ("a", "an") else int(qty)
    mult = {"minute": 1/1440, "hour": 1/24, "day": 1, "week": 7,
             "month": 30, "year": 365}[unit]
    return qty * mult

df["days_ago"] = df["review_time"].apply(relative_to_days)

# ---------------------------------------------------------------
# 5. Clean review text
#    - fix mojibake artifacts (ï¿½ , ý  from bad cp1252/utf8 decode)
#    - strip control chars, normalise whitespace
# ---------------------------------------------------------------
def clean_text(t):
    t = str(t)
    t = t.replace("\\n", " ")
    # remove replacement-character mojibake sequences
    t = re.sub(r"[ï¿½ý]+", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

df["review_clean"] = df["review"].apply(clean_text)

# ---------------------------------------------------------------
# 6. Drop rows with unusable review text (empty after cleaning, or
#    extremely short - not enough signal for sentiment analysis)
# ---------------------------------------------------------------
before = len(df)
df = df[df["review_clean"].str.len() >= 3].copy()
print(f"Dropped {before - len(df)} rows with unusable/empty review text")

# ---------------------------------------------------------------
# 6b. IMPORTANT DATA-QUALITY FINDING:
#     When a Google Maps reviewer leaves a star rating with NO written
#     comment, Google auto-fills the "review" field with a generic tag
#     that simply mirrors the star rating (5=Excellent, 4=Good,
#     3=Neutral, 2=Poor, 1=Terrible). These are NOT free-text opinions
#     and would trivially leak the label into any NLP model, so they
#     are flagged separately and excluded from the text-modelling set.
# ---------------------------------------------------------------
GENERIC_TAGS = {"excellent", "good", "neutral", "poor", "terrible",
                 "ok", "okay", "fine", "average"}
df["is_autotag"] = df["review_clean"].str.lower().str.strip().isin(GENERIC_TAGS)
print(f"Flagged {df['is_autotag'].sum()} auto-generated rating tags (no real text)")

# ---------------------------------------------------------------
# 7. Remove exact duplicate reviews (identical text + rating, most
#    likely re-scraped/duplicated rows)
# ---------------------------------------------------------------
before = len(df)
df = df.drop_duplicates(subset=["review_clean", "rating_num"]).copy()
print(f"Dropped {before - len(df)} duplicate reviews")

# ---------------------------------------------------------------
# 8. Derive sentiment label from star rating (ground-truth proxy)
#    1-2 stars -> Negative | 3 stars -> Neutral | 4-5 stars -> Positive
# ---------------------------------------------------------------
def rating_to_sentiment(r):
    if r <= 2:
        return "Negative"
    elif r == 3:
        return "Neutral"
    else:
        return "Positive"

df["sentiment"] = df["rating_num"].apply(rating_to_sentiment)

# ---------------------------------------------------------------
# Save
# ---------------------------------------------------------------
df = df.rename(columns={"latitude ": "latitude"})
keep_cols = ["reviewer_id", "store_name", "category", "store_address",
             "latitude", "longitude", "rating_count_num", "review_time",
             "days_ago", "review_clean", "rating_num", "sentiment",
             "is_autotag"]
df[keep_cols].to_csv(OUT_PATH, index=False)

print("\nFinal shape:", df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())
print("\nRating distribution:")
print(df["rating_num"].value_counts().sort_index())
print(f"\nSaved cleaned data -> {OUT_PATH}")


Raw shape: (33396, 10)
Dropped 179 rows with unusable/empty review text
Flagged 5167 auto-generated rating tags (no real text)
Dropped 10741 duplicate reviews

Final shape: (22476, 16)

Sentiment distribution:
sentiment
Positive    9819
Negative    9463
Neutral     3194
Name: count, dtype: int64

Rating distribution:
rating_num
1    7192
2    2271
3    3194
4    3635
5    6184
Name: count, dtype: int64

Saved cleaned data -> cleaned_reviews.csv
